# Human accessibility results: figures and LaTeX values

Run all cells from the repository root or from this notebook's directory. Change `INPUT_JSON` if your results file is elsewhere. The plots use the scores in the JSON: four WCAG categories and the unscaled JSON field `overall_score_only_special` for **Special Needs**. The boxplots do not include Overall. It also prints `overall_score_no_special / 3` for use in LaTeX.

These results use a 0–1 score scale. The example screenshots have negative scores, so their coordinates cannot be reproduced from this JSON without a separate score transformation. Records with missing, nonnumeric, or out-of-range category scores are listed and excluded from all plotted summaries so the figures use the same set of KGs. Raw score ranges are printed first.

Install dependencies in the notebook environment if needed: `%pip install pandas seaborn matplotlib` (then restart the kernel).

In [ ]:
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

INPUT_JSON = Path('supplementary_material/raw_kgheartbeat_results.json')
OUTPUT_DIR = Path('plots')
if not INPUT_JSON.exists():
    INPUT_JSON = Path('HumanAccessibility_LODCloud_results.json')
if not INPUT_JSON.exists():
    raise FileNotFoundError('Set INPUT_JSON to the results JSON path.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='paper', font_scale=1.05)
print(f'Input: {INPUT_JSON.resolve()}')
print(f'Figures and tables: {OUTPUT_DIR.resolve()}')

## Read and check the data

`Special Needs` uses `overall_score_only_special` directly. The extra no-special overall statistic uses `overall_score_no_special / 3`. The two individual accessibility fields remain available for the radar chart. A record is retained only when all six component scores are finite and between 0 and 1. The raw and retained counts below make this choice auditable.

In [ ]:
SCORE_FIELDS = {
    'Robust': 'robust_score',
    'Perceivable': 'perceivable_score',
    'Understandable': 'understandable_score',
    'Operable': 'operable_score',
    'Special Needs': 'overall_score_only_special',
}
COMPONENT_FIELDS = [
    'robust_score', 'perceivable_score', 'understandable_score', 'operable_score',
    'accessibility_for_visually_impaired_score',
    'accessibility_for_deaf_hearing_score',
]
RADAR_FIELDS = {
    'Perceivable': 'perceivable_score',
    'Accessibility for Deaf': 'accessibility_for_deaf_hearing_score',
    'Accessibility for Visually Impaired': 'accessibility_for_visually_impaired_score',
    'Operable': 'operable_score',
    'Understandable': 'understandable_score',
    'Robust': 'robust_score',
}
CATEGORY_ORDER = list(SCORE_FIELDS)
DOMAIN_ORDER = [
    'social_networking', 'geography', 'life_sciences', 'government',
    'user_generated', 'cross_domain', 'publications', 'media',
    'linguistics', '',
]

with INPUT_JSON.open(encoding='utf-8') as handle:
    results = json.load(handle)
if not isinstance(results, dict):
    raise TypeError('Expected a JSON object mapping KG identifiers to result objects.')

rows = []
for kg_id, item in results.items():
    if not isinstance(item, dict):
        continue
    row = {'kg_id': kg_id, 'domain': item.get('domain') or 'Unspecified'}
    for field in set(COMPONENT_FIELDS + list(SCORE_FIELDS.values()) + ['overall_score_no_special', 'overall_score_with_special']):
        value = item.get(field)
        row[field] = value if isinstance(value, (int, float)) and not isinstance(value, bool) else np.nan
    rows.append(row)
raw = pd.DataFrame(rows).set_index('kg_id')
check_fields = COMPONENT_FIELDS
sum_fields = ['overall_score_no_special', 'overall_score_with_special', 'overall_score_only_special']
valid = raw[check_fields + sum_fields].notna().all(axis=1) & np.isfinite(raw[check_fields + sum_fields]).all(axis=1)
valid &= raw[check_fields].ge(0).all(axis=1) & raw[check_fields].le(1).all(axis=1)
clean = raw.loc[valid].copy()
clean['Special Needs'] = clean['overall_score_only_special']
clean['Overall No Special'] = clean['overall_score_no_special'] / 3
sum_checks = {
    'overall_score_no_special': COMPONENT_FIELDS[:4],
    'overall_score_with_special': COMPONENT_FIELDS,
    'overall_score_only_special': COMPONENT_FIELDS[4:],
}
for total, parts in sum_checks.items():
    if clean[parts].sum(axis=1).sub(clean[total]).abs().gt(1e-8).any():
        raise ValueError(f'{total} disagrees with its component scores.')
if not clean[['Special Needs', 'Overall No Special']].ge(0).all().all():
    raise ValueError('A category or normalized aggregate score is negative.')
if clean.empty:
    raise ValueError('No valid records remain after score checks.')

print(f'JSON entries: {len(results)}; result objects: {len(raw)}; retained: {len(clean)}; excluded: {(~valid).sum()}')
print('Excluded KG IDs:', ', '.join(raw.index[~valid]) or '(none)')
raw_ranges = raw[check_fields + sum_fields].agg(['count', 'min', 'max']).T
display(raw_ranges.round(6))
clean['Domain'] = clean['domain'].replace({'': 'Unspecified'}).fillna('Unspecified')
domain_order = [d.replace('_', ' ').title() if d else 'Unspecified' for d in DOMAIN_ORDER]
clean['Domain'] = clean['Domain'].str.replace('_', ' ', regex=False).str.title()
domain_order += sorted(set(clean['Domain']) - set(domain_order))
print('Retained records by domain:')
display(clean['Domain'].value_counts().reindex(domain_order, fill_value=0).rename('N').to_frame())

## Boxplots by domain

One panel per domain, with the five requested category scores. Special Needs uses `overall_score_only_special` directly. The `Unspecified` panel contains records whose domain is blank.

In [ ]:
plot_frame = clean[['Domain', 'Special Needs', *list(SCORE_FIELDS.values())[:4]]].rename(
    columns={source: label for label, source in SCORE_FIELDS.items() if label != 'Special Needs'}
)
long = plot_frame.melt(id_vars='Domain', value_vars=CATEGORY_ORDER,
                       var_name='Category', value_name='Score')
active_domains = [name for name in domain_order if name in set(long['Domain'])]
g = sns.catplot(
    data=long, kind='box', x='Score', y='Category', col='Domain',
    col_order=active_domains, col_wrap=5, order=CATEGORY_ORDER,
    color='#3989ee', whis=1.5, fliersize=2, linewidth=0.9,
    height=2.5, aspect=1.15, sharex=True, sharey=True,
)
g.set(xlim=(-0.02, 1.02), xlabel='Score', ylabel='Category')
g.set_titles('{col_name}')
for ax in g.axes.flat:
    ax.axvline(0, color='#555555', linestyle=':', linewidth=0.8)
g.figure.subplots_adjust(top=0.89, hspace=0.48)
g.figure.suptitle('Human accessibility scores by LOD Cloud domain')
for suffix in ('png', 'pdf'):
    g.figure.savefig(OUTPUT_DIR / f'boxplots_by_domain.{suffix}', dpi=300, bbox_inches='tight')
plt.show()

## LaTeX boxplots (PGFPlots)

Run this cell after the Python boxplot cell. It writes `human_accessibility_figures/boxplots_by_domain.tex`, a standalone, two-row PGFPlots figure in the style of the supplied example. The values come from the retained 0–1 JSON scores; whiskers use the 1.5×IQR rule and points outside them are plotted as outliers. To compile, run `pdflatex -output-directory human_accessibility_figures human_accessibility_figures/boxplots_by_domain.tex` from the repository root. You need the LaTeX `standalone`, `tikz`, and `pgfplots` packages. The earlier cell also saves a PDF directly from Matplotlib.


In [ ]:
# Create an editable LaTeX figure from the same observations as the Python plot.
def tex_label(value):
    return (str(value).replace('\\', r'\textbackslash{}')
            .replace('&', r'\&').replace('%', r'\%')
            .replace('_', r'\_').replace('#', r'\#'))

tex_lines = [
    r'\documentclass[tikz,border=5pt]{standalone}',
    r'\usepackage{pgfplots}',
    r'\usepgfplotslibrary{statistics}',
    r'\pgfplotsset{compat=1.18}',
    r'\begin{document}',
    r'\begin{tikzpicture}',
]
category_ticks = ','.join(tex_label(name) for name in CATEGORY_ORDER)
for panel, domain in enumerate(active_domains):
    column, row = panel % 5, panel // 5
    axis_options = [
        f'at={{({column * 6.5:.1f}cm,{-row * 6.0:.1f}cm)}}',
        'anchor=north west', 'width=5.0cm', 'height=4.0cm',
        'boxplot/draw direction=x',
        'xmin=-0.02', 'xmax=1.02', 'ymin=0.5', 'ymax=5.5', 'y dir=reverse',
        'xtick={0,0.25,0.5,0.75,1}', 'ytick={1,2,3,4,5}',
        f'title={{{tex_label(domain)}}}',
    ]
    if column == 0:
        axis_options += [f'yticklabels={{{category_ticks}}}', 'ylabel={Category}']
    else:
        axis_options.append('yticklabels={}')
    if row == 1:
        axis_options.append('xlabel={Score}')
    tex_lines.append(r'\begin{axis}[' + ',\n  '.join(axis_options) + ']')
    for position, category in enumerate(CATEGORY_ORDER, start=1):
        values = np.sort(long.loc[
            long['Domain'].eq(domain) & long['Category'].eq(category), 'Score'
        ].to_numpy(dtype=float))
        q1, median, q3 = np.quantile(values, [0.25, 0.5, 0.75])
        iqr = q3 - q1
        inside = values[(values >= q1 - 1.5 * iqr) & (values <= q3 + 1.5 * iqr)]
        lower, upper = inside[0], inside[-1]
        stats = dict(zip(
            ['lower whisker', 'lower quartile', 'median', 'upper quartile', 'upper whisker'],
            [lower, q1, median, q3, upper]
        ))
        prepared = ', '.join(f'{key}={value:.6f}' for key, value in stats.items())
        tex_lines.append(
            r'\addplot+[boxplot prepared={' + prepared +
            f'}}, boxplot/draw position={position}, fill=blue!35, '
            r'draw=blue!60!black] coordinates {};'
        )
        outliers = values[(values < lower) | (values > upper)]
        if len(outliers):
            coordinates = ' '.join(f'({value:.6f},{position})' for value in outliers)
            tex_lines.append(
                r'\addplot+[only marks, mark=*, mark size=0.8pt, '
                r'draw=blue!60!black, forget plot] coordinates {' + coordinates + '};'
            )
    tex_lines.append(r'\end{axis}')
tex_lines += [r'\end{tikzpicture}', r'\end{document}']
tex_path = OUTPUT_DIR / 'boxplots_by_domain.tex'
tex_path.write_text('\n'.join(tex_lines) + '\n', encoding='utf-8')
print(f'Saved editable LaTeX boxplots to {tex_path.resolve()}')
print(f'Compile with: pdflatex -output-directory {OUTPUT_DIR} {tex_path}')


## Mean radar chart

The radial chart shows the arithmetic mean of each underlying category, including the two separate accessibility categories.

In [ ]:
radar_means = clean[list(RADAR_FIELDS.values())].mean().rename(
    index={source: label for label, source in RADAR_FIELDS.items()}
)
labels = list(RADAR_FIELDS)
values = radar_means.reindex(labels).to_numpy()
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False)
fig, ax = plt.subplots(figsize=(7, 6), subplot_kw={'projection': 'polar'})
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.plot(np.r_[angles, angles[0]], np.r_[values, values[0]], color='#3989ee', linewidth=2)
ax.fill(np.r_[angles, angles[0]], np.r_[values, values[0]], color='#3989ee', alpha=0.4)
ax.set_xticks(angles, labels)
ax.set_ylim(0, 1)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_title('Mean accessibility scores', pad=28)
fig.tight_layout()
for suffix in ('png', 'pdf'):
    fig.savefig(OUTPUT_DIR / f'mean_radar.{suffix}', dpi=300, bbox_inches='tight')
plt.show()

# Export an editable TikZ radar chart using the same means and axis order.
radar_tex = [
    r'\documentclass[tikz,border=6pt]{standalone}',
    r'\definecolor{scoreblue}{HTML}{3989EE}',
    r'\begin{document}',
    r'\begin{tikzpicture}[font=\small]',
    f'% Retained datasets: {len(clean)}. Radial scores range from 0 to 1.',
]
for tick in (0.25, 0.5, 0.75, 1.0):
    radar_tex.append(r'\draw[gray!35] (0,0) circle[radius=' + f'{4 * tick:.6f}cm];')
    radar_tex.append(r'\node[anchor=west,fill=white,inner sep=1pt] at (5:' + f'{4 * tick:.6f}' + ') {' + f'{tick:g}' + '};')
radar_coordinates = []
for index, (label, value) in enumerate(zip(labels, values)):
    angle = 90 - index * 360 / len(labels)
    radar_tex.append(r'\draw[gray!35] (0,0) -- (' + f'{angle:.6f}:4);')
    anchor = 'south' if index == 0 else 'north' if index == len(labels) // 2 else 'west' if angle > -90 else 'east'
    radar_tex.append(r'\node[anchor=' + anchor + r',align=center,text width=3.3cm] at (' + f'{angle:.6f}:4.3' + ') {' + tex_label(label) + '};')
    radar_coordinates.append(f'({angle:.6f}:{4 * value:.12g})')
radar_tex.append(r'\path[draw=scoreblue,fill=scoreblue,fill opacity=0.4,draw opacity=1,line width=1.2pt] ' + ' -- '.join(radar_coordinates) + ' -- cycle;')
radar_tex.append(r'\node[font=\large] at (0,5.3) {Mean accessibility scores};')
radar_tex.extend([r'\end{tikzpicture}', r'\end{document}'])
radar_tex_path = OUTPUT_DIR / 'mean_radar.tex'
radar_tex_path.write_text('\n'.join(radar_tex) + '\n', encoding='utf-8')
print(f'Saved editable radar chart to {radar_tex_path.resolve()}')


## Radar chart by domain, including All

Each ray represents a domain; each colored series represents one of the five accessibility categories in the supplied radar layout. **All** is the arithmetic mean across all retained datasets, including those with an unspecified domain. It is computed from individual datasets, so larger domains contribute proportionally to their dataset counts.

The chart uses the current JSON scores, starting at zero, rather than the negative example values. Special Needs uses `overall_score_only_special` directly. The scale extends above 1 if needed to accommodate this sum. Additional domains, including Unspecified, receive their own rays. The PNG, PDF, editable TikZ figure, and mean/count table are saved as `domain_radar.*` and `domain_radar_statistics.csv`.


In [ ]:
# Domain rays and one pooled ray, with a series for each accessibility category.
domain_radar_frame = clean[['Domain', *SCORE_FIELDS.values()]].rename(
    columns={source: label for label, source in SCORE_FIELDS.items()}
)
domain_radar_order = [
    'Government', 'Geography', 'Cross Domain', 'User Generated',
    'Social Networking', 'Publications', 'Media', 'Linguistics', 'Life Sciences',
]
present_domains = set(domain_radar_frame['Domain'])
domain_radar_order = [name for name in domain_radar_order if name in present_domains]
domain_radar_order += sorted(present_domains - set(domain_radar_order))
domain_radar_means = domain_radar_frame.groupby('Domain')[CATEGORY_ORDER].mean().reindex(domain_radar_order)
# Compute directly from datasets, rather than averaging the domain means.
domain_radar_means.loc['All'] = domain_radar_frame[CATEGORY_ORDER].mean()
domain_radar_counts = domain_radar_frame.groupby('Domain').size().reindex(domain_radar_order)
domain_radar_counts.loc['All'] = len(domain_radar_frame)
domain_radar_labels = list(domain_radar_means.index)
domain_radar_angles = np.linspace(0, 2 * np.pi, len(domain_radar_labels), endpoint=False)
domain_radar_colors = {
    'Robust': '#245cbb', 'Perceivable': '#00a6b2',
    'Understandable': '#409b46', 'Operable': '#dc495b', 'Special Needs': '#ed9c36',
}
# Special Needs remains the original sum of two scores, as in the other figures.
domain_radar_limit = max(1.0, math.ceil(float(domain_radar_means.max().max()) * 5) / 5)
fig, ax = plt.subplots(figsize=(10, 9), subplot_kw={'projection': 'polar'})
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
for category in CATEGORY_ORDER:
    series = domain_radar_means[category].to_numpy()
    closed_angles = np.r_[domain_radar_angles, domain_radar_angles[0]]
    closed_values = np.r_[series, series[0]]
    ax.plot(closed_angles, closed_values, color=domain_radar_colors[category],
            linewidth=1.8, label=category)
    ax.fill(closed_angles, closed_values, color=domain_radar_colors[category], alpha=0.1)
ax.set_xticks(domain_radar_angles, domain_radar_labels, fontsize=11, fontweight='bold')
ax.tick_params(axis='x', pad=18)
ax.set_ylim(0, domain_radar_limit)
ax.set_yticks(np.linspace(0, domain_radar_limit, 6))
ax.grid(linestyle='--', alpha=0.5)
ax.set_title('Mean accessibility scores by domain and across all datasets', pad=35)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
fig.tight_layout()
for suffix in ('png', 'pdf'):
    fig.savefig(OUTPUT_DIR / f'domain_radar.{suffix}', dpi=300, bbox_inches='tight')
plt.show()

# Export editable TikZ using the same means, ordering, and radial scale.
domain_radar_tex = [
    r'\documentclass[tikz,border=8pt]{standalone}',
    r'\begin{document}',
    r'\begin{tikzpicture}[font=\small]',
    f'% Retained datasets: {len(clean)}. All is the pooled dataset mean.',
]
for index, category in enumerate(CATEGORY_ORDER):
    domain_radar_tex.append(r'\definecolor{domainseries' + str(index) + '}{HTML}{' + domain_radar_colors[category][1:] + '}')
for tick in np.linspace(0, domain_radar_limit, 6):
    radius = 4 * tick / domain_radar_limit
    domain_radar_tex.append(r'\draw[gray!40,densely dashed] (0,0) circle[radius=' + f'{radius:.6f}cm];')
    domain_radar_tex.append(r'\node[fill=white,inner sep=1pt] at (80:' + f'{radius:.6f}' + ') {' + f'{tick:g}' + '};')
for index, label in enumerate(domain_radar_labels):
    angle = 90 - index * 360 / len(domain_radar_labels)
    domain_radar_tex.append(r'\draw[gray!50] (0,0) -- (' + f'{angle:.6f}:4.1);')
    anchor = 'south' if abs(angle - 90) < 1e-8 else 'north' if abs(angle + 90) < 1e-8 else 'west' if angle > -90 else 'east'
    domain_radar_tex.append(r'\node[anchor=' + anchor + r',font=\small\bfseries] at (' + f'{angle:.6f}:4.5' + ') {' + tex_label(label) + '};')
for index, category in enumerate(CATEGORY_ORDER):
    coordinates = [
        f'({90 - position * 360 / len(domain_radar_labels):.6f}:{4 * value / domain_radar_limit:.12g})'
        for position, value in enumerate(domain_radar_means[category])
    ]
    color = f'domainseries{index}'
    domain_radar_tex.append(r'\path[draw=' + color + ',fill=' + color + r',fill opacity=0.1,line width=1pt] ' + ' -- '.join(coordinates) + ' -- cycle;')
    column, row = index % 3, index // 3
    x, y = -5 + column * 3.8, -5.8 - row * 0.6
    domain_radar_tex.append(r'\draw[draw=' + color + ',fill=' + color + ',fill opacity=0.3] ' + f'({x:.2f},{y:.2f}) rectangle ({x + 0.4:.2f},{y + 0.25:.2f});')
    domain_radar_tex.append(r'\node[anchor=west] at (' + f'{x + 0.5:.2f},{y + 0.125:.2f}' + ') {' + tex_label(category) + '};')
domain_radar_tex.append(r'\node[font=\large,align=center] at (0,5.5) {Mean accessibility scores by domain\\and across all datasets};')
domain_radar_tex.extend([r'\end{tikzpicture}', r'\end{document}'])
(OUTPUT_DIR / 'domain_radar.tex').write_text('\n'.join(domain_radar_tex) + '\n', encoding='utf-8')
domain_radar_table = domain_radar_means.copy()
domain_radar_table.insert(0, 'N', domain_radar_counts.astype(int))
domain_radar_table.to_csv(OUTPUT_DIR / 'domain_radar_statistics.csv', float_format='%.6f')
display(domain_radar_table.round(6))
print(f'Saved domain radar PNG, PDF, LaTeX, and statistics to {OUTPUT_DIR.resolve()}')


## Score distributions

Special Needs is split into `accessibility_for_visually_impaired_score` and `accessibility_for_deaf_hearing_score`, each shown separately.

The Overall category uses each dataset's `overall_score_with_special / 6`, normalized to [0, 1].

Blue violins show the score distribution, with each category scaled to the same maximum width. This prevents the sharp peaks near zero from flattening the other categories under area normalization. Widths show relative density within each category, not density comparisons between categories. Red bars mark medians. A constant distribution is shown as a blue point.

Requires Seaborn 0.13 or newer (`%pip install --upgrade seaborn` if needed). The other category scores are unchanged; near-zero ranges can still appear narrow on the score axis.


In [ ]:
violin_fields = {
    **{label: field for label, field in SCORE_FIELDS.items() if label != 'Special Needs'},
    'Accessibility for Visually Impaired': 'accessibility_for_visually_impaired_score',
    'Accessibility for Deaf/Hearing': 'accessibility_for_deaf_hearing_score',
}
violin_order = [*violin_fields, 'Overall']
violin_frame = clean[['Domain', *violin_fields.values()]].rename(
    columns={field: label for label, field in violin_fields.items()}
)
violin_frame['Overall'] = clean['overall_score_with_special'] / 6
violin_long = violin_frame.melt(id_vars='Domain', value_vars=violin_order,
                                var_name='Category', value_name='Score')

fig, ax = plt.subplots(figsize=(11, 6))
varying = [name for name in violin_order if violin_long.loc[violin_long['Category'].eq(name), 'Score'].nunique() > 1]
sns.violinplot(data=violin_long[violin_long['Category'].isin(varying)], x='Score', y='Category',
               order=violin_order, color='#3989ee', saturation=1,
               inner=None, cut=0, density_norm='width', common_norm=False,
               linewidth=0.9, linecolor='#2161a8', ax=ax)
summary = violin_long.groupby('Category')['Score'].agg(['count', 'mean', 'median', 'min', 'max'])
for i, category in enumerate(violin_order):
    stats = summary.loc[category]
    if stats['min'] == stats['max']:
        ax.scatter(stats['min'], i, s=90, color='#3989ee', zorder=4)
    ax.plot(stats['median'], i, marker='|', color='red', markersize=18, markeredgewidth=1.6)
ax.set_xlim(-0.02, 1.02)
ax.set(xlabel='Score', ylabel='', title='Score distributions by accessibility category')
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#3989ee', edgecolor='#2161a8', label='Distribution (equal maximum width)'),
    Line2D([], [], marker='|', color='red', linestyle='None', markersize=12, label='Median'),
], loc='lower center', bbox_to_anchor=(0.5, 1.1), ncol=2, frameon=False)
fig.tight_layout()
for suffix in ('png', 'pdf'):
    fig.savefig(OUTPUT_DIR / f'score_distributions.{suffix}', dpi=300, bbox_inches='tight')
plt.show()

# Export the actual violin outlines, preserving the plotted KDE and scaling.
from matplotlib.collections import PolyCollection
violin_tex = [
    r'\documentclass[tikz,border=6pt]{standalone}',
    r'\usepackage{pgfplots}',
    r'\pgfplotsset{compat=1.18}',
    r'\definecolor{scoreblue}{HTML}{3989EE}',
    r'\definecolor{scoreedge}{HTML}{2161A8}',
    r'\begin{document}',
    r'\begin{tikzpicture}',
    f'% Retained datasets: {len(clean)}. Each violin has equal maximum width.',
    r'\begin{axis}[width=17cm,height=9cm,',
    'xmin=-0.02,xmax=' + f'{ax.get_xlim()[1]:.12g}' + ',',
    'ymin=-0.5,ymax=' + f'{len(violin_order) - 0.5}' + ',y dir=reverse,',
    'ytick={' + ','.join(str(i) for i in range(len(violin_order))) + '},',
    'yticklabels={' + ','.join(tex_label(name) for name in violin_order) + '},',
    r'xlabel={Score},title={Score distributions by accessibility category},',
    r'xmajorgrids=true,ymajorgrids=false,grid style={gray!30},',
    r'legend style={at={(0.5,1.15)},anchor=south,legend columns=2,draw=none}]',
]
for body in ax.collections:
    if not isinstance(body, PolyCollection):
        continue
    for path in body.get_paths():
        coordinates = ' '.join(f'({x:.12g},{y:.12g})' for x, y in path.vertices)
        violin_tex.append(r'\addplot[draw=scoreedge,fill=scoreblue,line width=0.6pt,mark=none,forget plot] coordinates {' + coordinates + r'} \closedcycle;')
for index, category in enumerate(violin_order):
    stats = summary.loc[category]
    median = stats['median']
    if stats['min'] == stats['max']:
        violin_tex.append(r'\addplot[only marks,mark=*,mark size=3pt,scoreblue,forget plot] coordinates {' + f'({median:.12g},{index})' + '};')
    violin_tex.append(r'\addplot[red,line width=1pt,mark=none,forget plot] coordinates {' + f'({median:.12g},{index - 0.14:.6f}) ({median:.12g},{index + 0.14:.6f})' + '};')
violin_tex.extend([
    r'\addlegendimage{area legend,draw=scoreedge,fill=scoreblue}',
    r'\addlegendentry{Distribution (equal maximum width)}',
    r'\addlegendimage{red,line width=1pt}',
    r'\addlegendentry{Median}',
    r'\end{axis}', r'\end{tikzpicture}', r'\end{document}',
])
violin_tex_path = OUTPUT_DIR / 'score_distributions.tex'
violin_tex_path.write_text('\n'.join(violin_tex) + '\n', encoding='utf-8')
print(f'Saved editable score distributions to {violin_tex_path.resolve()}')


## Numbers to copy into LaTeX

The tables give N, mean, median, minimum, maximum, and quartiles. The first table supports the plotted distribution figure, the second the radar chart, and the CSV file gives the same statistics for each domain panel. A separate raw table includes every numeric JSON value, including unusual values excluded from plots. `Overall No Special` is printed separately using the requested divisor of 3. LaTeX rows and radar coordinates are printed below.

In [ ]:
def describe_scores(frame, group_columns):
    grouped = frame.groupby(group_columns, observed=True)['Score']
    table = grouped.agg(n='count', mean='mean', median='median', minimum='min', maximum='max')
    table['q1'] = grouped.quantile(0.25)
    table['q3'] = grouped.quantile(0.75)
    return table[['n', 'mean', 'median', 'minimum', 'maximum', 'q1', 'q3']]

category_stats = describe_scores(long, 'Category').reindex(CATEGORY_ORDER)
domain_stats = describe_scores(long, ['Domain', 'Category']).reindex(
    pd.MultiIndex.from_product([active_domains, CATEGORY_ORDER], names=['Domain', 'Category'])
)
raw_plot = raw[['overall_score_no_special', *SCORE_FIELDS.values()]].copy()
raw_plot['Special Needs'] = raw_plot['overall_score_only_special']
raw_plot['Overall No Special'] = raw_plot['overall_score_no_special'] / 3
raw_plot = raw_plot.rename(columns={source: label for label, source in SCORE_FIELDS.items() if label != 'Special Needs'})
raw_long = raw_plot.melt(value_vars=CATEGORY_ORDER, var_name='Category', value_name='Score')
raw_category_stats = describe_scores(raw_long.dropna(subset=['Score']), 'Category').reindex(CATEGORY_ORDER)
no_special_long = clean[['Domain', 'Overall No Special']].rename(columns={'Overall No Special': 'Score'})
no_special_stats = describe_scores(no_special_long.assign(Category='Overall No Special'), 'Category')
no_special_domain_stats = describe_scores(no_special_long, 'Domain').reindex(active_domains)
raw_no_special_stats = describe_scores(
    raw_plot[['Overall No Special']].rename(columns={'Overall No Special': 'Score'}).dropna().assign(Category='Overall No Special'), 'Category'
)
radar_stats = clean[list(RADAR_FIELDS.values())].agg(['count', 'mean', 'median', 'min', 'max']).T
radar_stats.index = list(RADAR_FIELDS)
print('Plotted records (valid 0–1 component scores):')
display(category_stats.round(4))
display(radar_stats.round(4))
display(domain_stats.round(4))
print('All numeric JSON scores (includes out-of-range values):')
display(raw_category_stats.round(4))
print('Overall No Special / 3: plotted records, then all numeric JSON values:')
display(no_special_stats.round(4))
display(raw_no_special_stats.round(4))
print('Overall No Special / 3 by domain:')
display(no_special_domain_stats.round(4))
raw_category_stats.to_csv(OUTPUT_DIR / 'raw_category_statistics.csv', float_format='%.6f')
no_special_stats.to_csv(OUTPUT_DIR / 'no_special_overall_statistics.csv', float_format='%.6f')
raw_no_special_stats.to_csv(OUTPUT_DIR / 'raw_no_special_overall_statistics.csv', float_format='%.6f')
no_special_domain_stats.to_csv(OUTPUT_DIR / 'no_special_overall_domain_statistics.csv', float_format='%.6f')
category_stats.to_csv(OUTPUT_DIR / 'category_statistics.csv', float_format='%.6f')
domain_stats.to_csv(OUTPUT_DIR / 'domain_category_statistics.csv', float_format='%.6f')
radar_stats.to_csv(OUTPUT_DIR / 'radar_statistics.csv', float_format='%.6f')

def latex_escape(value):
    return str(value).replace('\\', r'\textbackslash{}').replace('_', r'\_').replace('&', r'\&')

print('LaTeX rows: category & N & mean & median & min & max & Q1 & Q3 ' + chr(92) * 2)
for name, row in category_stats.iterrows():
    nums = ' & '.join([str(int(row['n']))] + [f'{row[key]:.4f}' for key in ['mean', 'median', 'minimum', 'maximum', 'q1', 'q3']])
    print(f'{latex_escape(name)} & {nums} ' + chr(92) * 2)
row = no_special_stats.loc['Overall No Special']
nums = ' & '.join([str(int(row['n']))] + [f'{row[key]:.4f}' for key in ['mean', 'median', 'minimum', 'maximum', 'q1', 'q3']])
print(f'Overall (No Special) & {nums} ' + chr(92) * 2)
print('\nRadar mean coordinates, in plotted order:')
print(' '.join(f'({i + 1},{radar_means[name]:.6f})' for i, name in enumerate(labels)))
print('\nExact radar means:')
for name in labels:
    print(f'{name}: {radar_means[name]:.6f}')
print(f'\nSaved figures and CSV tables to {OUTPUT_DIR.resolve()}')

## Boxplots of dimensions within each category

These figures pool datasets across domains and put categories in panels, with one horizontal boxplot per underlying metric. The first has five panels, merging the visual and hearing dimensions into Special Needs. The second has six panels, separating Visually Impaired and Deaf/Hearing. Shared dimensions appear once in the combined panel and in each applicable separate panel. The evaluator references video descriptions twice in its hearing score; this figure shows that dimension only once.

Dimensions follow the category formulas in `_evaluator_scores`. Each metric uses the first number in its JSON result. The dataset cohort matches the other charts; missing, nonnumeric, nonfinite, or out-of-range metric values are omitted per dimension and reported below. Whiskers use 1.5×IQR, with outliers shown as points; scores remain on [0, 1]. Each figure is exported as PNG, PDF, and editable PGFPlots LaTeX, with a CSV of per-dimension statistics and sample counts.


In [ ]:
# Use the dimensions that contribute to the evaluator's category formulas.
DIMENSION_FIELDS = {
    'Robust': ['version', 'webpage_status', 'metadata_broken_links_rate', 'canonical_citation'],
    'Perceivable': ['image_metadata', 'image', 'audio_meta', 'audio', 'video_meta', 'video'],
    'Understandable': ['metadata_lang', 'data_lang', 'human_redeable_labels', 'example', 'description_readability'],
    'Operable': ['contact_point', 'dump_size', 'sparql_availability_evaluator', 'common_format_availability', 'open_license', 'alternative_access_point'],
    'Visually Impaired': ['alt_image', 'audio_description_ratio'],
    'Deaf/Hearing': ['video_description_ratio', 'video_sign_language_ratio', 'audio_sign_language_ratio', 'video_subtitle_ratio', 'audio_subtitle_ratio', 'audio_description_ratio'],
}
DIMENSION_LABELS = {
    'version': 'Version information', 'webpage_status': 'Webpage availability',
    'metadata_broken_links_rate': 'Working metadata links', 'canonical_citation': 'Canonical citation',
    'image_metadata': 'Image metadata', 'image': 'Images', 'audio_meta': 'Audio metadata',
    'audio': 'Audio', 'video_meta': 'Video metadata', 'video': 'Video',
    'metadata_lang': 'Metadata language', 'data_lang': 'Data language',
    'human_redeable_labels': 'Human-readable labels', 'example': 'Examples',
    'description_readability': 'Description readability', 'contact_point': 'Contact point',
    'dump_size': 'Dump size', 'sparql_availability_evaluator': 'SPARQL availability',
    'common_format_availability': 'Common formats', 'open_license': 'Open license',
    'alternative_access_point': 'Alternative access point', 'alt_image': 'Image alternative text',
    'audio_description_ratio': 'Audio descriptions', 'video_description_ratio': 'Video descriptions',
    'video_sign_language_ratio': 'Video sign language', 'audio_sign_language_ratio': 'Audio sign language',
    'video_subtitle_ratio': 'Video subtitles', 'audio_subtitle_ratio': 'Audio subtitles',
}
# Keep the same dataset cohort as the existing charts; validate each metric separately.
dimension_values = {}
dimension_exclusions = {}
for field in dict.fromkeys(field for fields in DIMENSION_FIELDS.values() for field in fields):
    values, excluded = [], []
    for kg_id in clean.index:
        metric = results[kg_id].get(field)
        value = metric[0] if isinstance(metric, (list, tuple)) and metric else None
        if (isinstance(value, (int, float)) and not isinstance(value, bool)
                and math.isfinite(value) and 0 <= value <= 1):
            values.append(value)
        else:
            excluded.append(kg_id)
    if not values:
        raise ValueError(f'No valid scores for dimension {field}.')
    dimension_values[field] = np.asarray(values, dtype=float)
    if excluded:
        dimension_exclusions[field] = excluded
print('Excluded missing, nonnumeric, nonfinite, or out-of-range dimension values:', dimension_exclusions or '(none)')

collapsed_dimensions = {category: fields for category, fields in DIMENSION_FIELDS.items()
                        if category not in ('Visually Impaired', 'Deaf/Hearing')}
# Merge the two dimension lists, retaining shared metrics only once.
collapsed_dimensions['Special Needs'] = list(dict.fromkeys(
    DIMENSION_FIELDS['Visually Impaired'] + DIMENSION_FIELDS['Deaf/Hearing']
))

def plot_category_dimensions(panels, stem, title):
    fig, axes = plt.subplots(2, 3, figsize=(19, 10), sharex=True)
    tex = [
        r'\documentclass[tikz,border=5pt]{standalone}',
        r'\usepackage{pgfplots}', r'\usepgfplotslibrary{statistics}',
        r'\pgfplotsset{compat=1.18}', r'\begin{document}', r'\begin{tikzpicture}',
        f'% Dataset cohort: {len(clean)}. Each row is an underlying dimension.',
    ]
    statistics_rows = []
    for panel, (category, fields) in enumerate(panels.items()):
        ax = axes.flat[panel]
        frame = pd.DataFrame([
            {'Dimension': DIMENSION_LABELS[field], 'Score': value}
            for field in fields for value in dimension_values[field]
        ])
        sns.boxplot(data=frame, x='Score', y='Dimension',
                    order=[DIMENSION_LABELS[field] for field in fields],
                    color='#3989ee', whis=1.5, fliersize=2, linewidth=0.9, ax=ax)
        ax.set(xlim=(-0.02, 1.02), xlabel='Score', ylabel='', title=category)
        ax.tick_params(axis='x', labelbottom=True)
        column, row = panel % 3, panel // 3
        options = [
            f'at={{({column * 11.5:.1f}cm,{-row * 8.0:.1f}cm)}}',
            'anchor=north west', 'width=6.5cm', 'height=6.5cm', 'scale only axis',
            'boxplot/draw direction=x', 'xmin=-0.02', 'xmax=1.02',
            f'ymin=0.5,ymax={len(fields) + 0.5}', 'y dir=reverse',
            'xtick={0,0.25,0.5,0.75,1}',
            'ytick={' + ','.join(str(i) for i in range(1, len(fields) + 1)) + '}',
            'yticklabels={' + ','.join(tex_label(DIMENSION_LABELS[field]) for field in fields) + '}',
            'yticklabel style={font=\\small}', f'title={{{tex_label(category)}}}',
            'xlabel={Score}', 'xmajorgrids=true', 'grid style={gray!30}',
        ]
        tex.append(r'\begin{axis}[' + ',\n  '.join(options) + ']')
        for position, field in enumerate(fields, start=1):
            values = np.sort(dimension_values[field])
            q1, median, q3 = np.quantile(values, [0.25, 0.5, 0.75])
            iqr = q3 - q1
            inside = values[(values >= q1 - 1.5 * iqr) & (values <= q3 + 1.5 * iqr)]
            lower, upper = inside[0], inside[-1]
            prepared = ', '.join(f'{key}={value:.12g}' for key, value in zip(
                ['lower whisker', 'lower quartile', 'median', 'upper quartile', 'upper whisker'],
                [lower, q1, median, q3, upper]))
            tex.append(r'\addplot+[boxplot prepared={' + prepared +
                       f'}},boxplot/draw position={position},fill=blue!35,draw=blue!60!black] coordinates {{}};')
            outliers = values[(values < lower) | (values > upper)]
            if len(outliers):
                coordinates = ' '.join(f'({value:.12g},{position})' for value in outliers)
                tex.append(r'\addplot+[only marks,mark=*,mark size=0.8pt,draw=blue!60!black,forget plot] coordinates {' + coordinates + '};')
            statistics_rows.append({
                'Category': category, 'Dimension': DIMENSION_LABELS[field], 'Field': field,
                'N': len(values), 'mean': values.mean(), 'median': median,
                'minimum': values[0], 'maximum': values[-1], 'q1': q1, 'q3': q3,
                'lower_whisker': lower, 'upper_whisker': upper,
                'excluded': len(clean) - len(values),
            })
        tex.append(r'\end{axis}')
    for ax in list(axes.flat)[len(panels):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=16)
    fig.tight_layout(rect=(0, 0, 1, 0.96), w_pad=3, h_pad=3)
    for suffix in ('png', 'pdf'):
        fig.savefig(OUTPUT_DIR / f'{stem}.{suffix}', dpi=300, bbox_inches='tight')
    plt.show()
    tex.extend([r'\end{tikzpicture}', r'\end{document}'])
    (OUTPUT_DIR / f'{stem}.tex').write_text('\n'.join(tex) + '\n', encoding='utf-8')
    table = pd.DataFrame(statistics_rows)
    table.to_csv(OUTPUT_DIR / f'{stem}_statistics.csv', index=False, float_format='%.6f')
    print(f'Saved {len(panels)} category panels: {stem} (PNG, PDF, LaTeX, CSV)')
    return table

collapsed_dimension_stats = plot_category_dimensions(
    collapsed_dimensions, 'boxplots_by_category_collapsed',
    'Accessibility dimensions by category — combined Special Needs',
)
separated_dimension_stats = plot_category_dimensions(
    DIMENSION_FIELDS, 'boxplots_by_category_separated',
    'Accessibility dimensions by category — separate visual and hearing accessibility',
)
